# 🏥 Medical Insurance Charges Prediction

**Only one folder is needed.** Put these 2 files in the same folder:
1. this notebook
2. your data file (`insurance.csv` or `insurance.xls`)

Open Jupyter inside that folder and run the cells one by one, from top to bottom.

## 1. Load the data

**Cell 1:** Install the extra libraries (only needed once; it is fine if they are already installed).

In [ ]:
%pip install -q streamlit plotly seaborn

**Cell 2:** Import libraries.

In [ ]:
import json, joblib, sys, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy import stats
import sklearn

from sklearn.base import clone
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

**Cell 3:** Load the data. This cell finds your file, fixes column names and checks that the data is correct.

In [ ]:
import os

def load_data():
    file = next((f for f in ["insurance.csv", "insurance.xls", "insurance.xlsx"] if os.path.exists(f)), None)
    if file is None:
        raise FileNotFoundError("Put insurance.csv in the same folder as this notebook. This notebook is running in: " + os.getcwd())
    readers = [lambda: pd.read_csv(file, sep=None, engine="python"), lambda: pd.read_excel(file)]
    for read in readers:
        try:
            data = read()
            data.columns = data.columns.astype(str).str.strip().str.lower()
            if "charges" in data.columns:
                return data
        except Exception:
            pass
    raise ValueError("Could not find the 'charges' column in " + file + ". Open the file and check the first row.")

df = load_data()
for c in ["sex", "smoker", "region"]:
    df[c] = df[c].astype(str).str.strip().str.lower()
df.head()

**Cell 4:** Shape of the data (rows, columns).

In [ ]:
df.shape

**Cell 5:** Column names, data types and non-null counts.

In [ ]:
df.info()

**Cell 6:** Check missing values.

In [ ]:
df.isnull().sum()

**Cell 7:** Check duplicate rows.

In [ ]:
df.duplicated().sum()

**Cell 8:** Remove the duplicate row (cleaning).

In [ ]:
df = df.drop_duplicates()
df.to_csv("insurance_clean.csv", index=False)
df.shape

**Cell 9:** Basic statistics.

In [ ]:
df.describe()

## 2. EDA (charts)

**Cell 10:** Distribution of charges, age and bmi.

In [ ]:
df[["charges", "age", "bmi"]].hist(bins=30, figsize=(12, 3), layout=(1, 3))
plt.show()

**Cell 11:** Count of each category.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, c in zip(ax, ["sex", "smoker", "region"]):
    sns.countplot(data=df, x=c, ax=a)
plt.show()

**Cell 12:** Charges by gender, smoker, region and children.

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 8))
for a, c in zip(ax.ravel(), ["sex", "smoker", "region", "children"]):
    sns.boxplot(data=df, x=c, y="charges", ax=a)
plt.show()

**Cell 13:** Scatter plots: features vs charges.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", ax=ax[0])
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", ax=ax[1])
sns.scatterplot(data=df, x="children", y="charges", ax=ax[2])
plt.show()

**Cell 14:** Correlation heatmap.

In [ ]:
sns.heatmap(df.select_dtypes("number").corr(), annot=True, cmap="coolwarm")
plt.show()

**Cell 15:** Boxplots to see outliers.

In [ ]:
df[["age", "bmi", "charges"]].plot(kind="box", subplots=True, figsize=(10, 4))
plt.show()

**Cell 16:** QQ plot of charges.

In [ ]:
stats.probplot(df["charges"], plot=plt)
plt.show()

**Cell 17:** One interactive Plotly chart (hover over it).

In [ ]:
px.scatter(df, x="bmi", y="charges", color="smoker", hover_data=["age", "sex", "region"])

## 3. Preprocessing

**Cell 18:** Split into X (inputs) and y (target), then train/test 80:20.

In [ ]:
X = df.drop("charges", axis=1)
y = df["charges"]

num_cols = X.select_dtypes("number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

**Cell 19:** Preprocessing: scale numbers, one-hot encode text. It is fitted only on training data inside the pipeline, so there is no data leakage.

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

## 4. Build the models

**Cell 20:** Create the 6 models. Each one is a pipeline (preprocessing + model). The dictionary on the right of each model is what GridSearchCV will try.

In [ ]:
def make(model, degree=None):
    steps = [("prep", clone(preprocessor))]
    if degree:
        steps.append(("poly", PolynomialFeatures(degree, include_bias=False)))
    steps.append(("model", model))
    return Pipeline(steps)

models = {
    "Linear Regression": (make(LinearRegression()), {}),
    "Polynomial (degree 2)": (make(LinearRegression(), 2), {}),
    "Polynomial (degree 3)": (make(LinearRegression(), 3), {}),
    "Decision Tree": (make(DecisionTreeRegressor(random_state=42)),
                      {"model__max_depth": [3, 4, 5, 6, 8], "model__min_samples_leaf": [5, 10, 20]}),
    "KNN": (make(KNeighborsRegressor()),
            {"model__n_neighbors": [5, 7, 9, 11, 15], "model__weights": ["uniform", "distance"]}),
    "Random Forest": (make(RandomForestRegressor(random_state=42)),
                      {"model__n_estimators": [100, 200], "model__max_depth": [4, 6, 8], "model__min_samples_leaf": [1, 3, 5]}),
}

**Cell 21:** Train all models on the training data (takes about a minute).

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
best_models = {}

for name, (pipe, grid) in models.items():
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
    gs.fit(X_train, y_train)
    best_models[name] = gs.best_estimator_

## 5. Evaluate the models

**Cell 22:** The evaluation function (MAE, MSE, RMSE, R²).

In [ ]:
def evaluate_regression(y_true, y_pred, model_name):
    mse = mean_squared_error(y_true, y_pred)
    return {"Model": model_name,
            "MAE": mean_absolute_error(y_true, y_pred),
            "MSE": mse,
            "RMSE": np.sqrt(mse),
            "R2 Score": r2_score(y_true, y_pred)}

**Cell 23:** Compare all models using cross-validation on the training data (the test data is not used here).

In [ ]:
rows = []
for name, model in best_models.items():
    cv_pred = cross_val_predict(model, X_train, y_train, cv=cv)
    row = evaluate_regression(y_train, cv_pred, name)
    row["CV RMSE"] = -cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error").mean()
    row["Train R2"] = model.score(X_train, y_train)
    rows.append(row)

results = pd.DataFrame(rows).sort_values("CV RMSE").reset_index(drop=True)
results

**Cell 24:** Pick the best model = lowest CV RMSE.

In [ ]:
best_name = results.loc[0, "Model"]
best_model = best_models[best_name]
best_name

**Cell 25:** Now test the best model on the test data (first time we use it).

In [ ]:
y_pred = best_model.predict(X_test)
test_scores = evaluate_regression(y_test, y_pred, best_name)
train_r2 = best_model.score(X_train, y_train)

pd.DataFrame([test_scores]).assign(Train_R2=train_r2)

## 6. Model charts

**Cell 26:** Actual vs predicted, residual plot, residual distribution and QQ plot of residuals.

In [ ]:
residuals = y_test - y_pred

fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].scatter(y_test, y_pred)
ax[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
ax[0].set_title("Actual vs Predicted")
ax[1].scatter(y_pred, residuals)
ax[1].axhline(0, color="r")
ax[1].set_title("Residual plot")
sns.histplot(residuals, kde=True, ax=ax[2])
ax[2].set_title("Residual distribution")
stats.probplot(residuals, plot=ax[3])
plt.show()

**Cell 27:** Model comparison chart (smaller bar = better).

In [ ]:
sns.barplot(data=results, x="CV RMSE", y="Model")
plt.show()

**Cell 28:** Feature importance (Random Forest).

In [ ]:
rf = best_models["Random Forest"]
names = rf.named_steps["prep"].get_feature_names_out()
pd.Series(rf.named_steps["model"].feature_importances_, index=names).sort_values().plot(kind="barh")
plt.show()

## 7. Save the model

**Cell 29:** Save the full pipeline (preprocessing + model) and the results.

In [ ]:
joblib.dump(best_model, "best_insurance_model.pkl")
results.to_csv("model_comparison.csv", index=False)
pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred}).to_csv("test_predictions.csv", index=False)

info = {"best_model": best_name,
        "cv_rmse": float(results.loc[0, "CV RMSE"]),
        "train_r2": float(train_r2),
        "test_r2": float(test_scores["R2 Score"]),
        "test_mae": float(test_scores["MAE"]),
        "test_mse": float(test_scores["MSE"]),
        "test_rmse": float(test_scores["RMSE"])}
json.dump(info, open("model_info.json", "w"), indent=2)

**Cell 30:** Test the saved model with raw input (plain text and numbers).

In [ ]:
saved = joblib.load("best_insurance_model.pkl")
new_person = pd.DataFrame([{"age": 35, "sex": "female", "bmi": 28.5, "children": 2, "smoker": "no", "region": "southwest"}])
saved.predict(new_person)

## 8. Streamlit app (inside this notebook)

The next cell saves `app.py` (5 pages: Home, Data Analysis, Model Performance, Prediction, About).

**Cell 31:** Create `app.py`. This only saves the file, it does not run the app yet.

In [ ]:
%%writefile app.py
# Medical Insurance Charges Prediction - simple Streamlit app
# Run:  streamlit run app.py
import json
from datetime import datetime
from pathlib import Path

import joblib
import pandas as pd
import plotly.express as px
import streamlit as st

BASE = Path(__file__).parent
st.set_page_config(page_title="Insurance Charges Prediction", page_icon="🏥", layout="wide")

# ---------- Simple styling ----------
st.markdown("""
<style>
.stApp {background: linear-gradient(135deg, #eef4ff, #f3ecff, #e8fbf8);}
[data-testid="stSidebar"] {background: linear-gradient(180deg, #1e3a8a, #6d28d9);}
[data-testid="stSidebar"] * {color: white !important;}
div[data-testid="stMetric"] {background: white; padding: 15px; border-radius: 15px;
    box-shadow: 0 4px 12px rgba(0,0,0,.08); border-left: 6px solid #7c3aed;}
.stButton > button, .stDownloadButton > button {background: linear-gradient(90deg, #2563eb, #7c3aed);
    color: white; border: none; border-radius: 10px; padding: .5rem 1.5rem;}
.title {background: linear-gradient(120deg, #2563eb, #7c3aed, #0d9488); color: white;
    padding: 25px; border-radius: 20px; font-size: 2rem; font-weight: 700; margin-bottom: 20px;}
.result {background: linear-gradient(120deg, #0d9488, #2563eb); color: white; padding: 25px;
    border-radius: 20px; text-align: center; font-size: 2.2rem; font-weight: 700; margin-top: 15px;}
.footer {text-align: center; color: #64748b; padding-top: 30px;}
</style>
""", unsafe_allow_html=True)

# ---------- Load files ----------
try:
    df = pd.read_csv(BASE / "insurance_clean.csv")
    model = joblib.load(BASE / "best_insurance_model.pkl")
    info = json.load(open(BASE / "model_info.json"))
    compare = pd.read_csv(BASE / "model_comparison.csv")
    preds = pd.read_csv(BASE / "test_predictions.csv")
except FileNotFoundError as e:
    st.error(f"File not found: {e.filename}. Please run the notebook first (it creates these files).")
    st.stop()

# ---------- Session state (for reset button and history) ----------
defaults = {"age": 30, "sex": "male", "bmi": 25.0, "children": 0, "smoker": "no", "region": "southwest"}
for k, v in defaults.items():
    st.session_state.setdefault("in_" + k, v)
st.session_state.setdefault("history", [])
st.session_state.setdefault("result", None)


def reset_form():
    for k, v in defaults.items():
        st.session_state["in_" + k] = v
    st.session_state["result"] = None


def title(text):
    st.markdown(f'<div class="title">{text}</div>', unsafe_allow_html=True)


def footer():
    st.markdown('<div class="footer">🏥 Medical Insurance Charges Prediction | Python, Scikit-learn, Streamlit</div>',
                unsafe_allow_html=True)


# ---------- Sidebar navigation ----------
page = st.sidebar.radio("Go to", ["🏠 Home", "📊 Data Analysis", "🤖 Model Performance",
                                  "💰 Prediction", "ℹ️ About Project"])

# ============ PAGE 1: HOME ============
if page == "🏠 Home":
    title("🏥 Medical Insurance Charges Prediction")
    st.write("**Objective:** predict yearly medical insurance charges from age, sex, BMI, children, "
             "smoker status and region, using machine learning.")
    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Total Records", len(df))
    c2.metric("Input Features", df.shape[1] - 1)
    c3.metric("Best Model", info["best_model"])
    c4.metric("Test R² Score", round(info["test_r2"], 3))
    c5, c6, c7 = st.columns(3)
    c5.metric("Test MAE ($)", f'{info["test_mae"]:,.0f}')
    c6.metric("Test RMSE ($)", f'{info["test_rmse"]:,.0f}')
    c7.metric("CV RMSE ($)", f'{info["cv_rmse"]:,.0f}')
    st.info("Use the sidebar on the left to open the other pages.")
    footer()

# ============ PAGE 2: DATA ANALYSIS ============
elif page == "📊 Data Analysis":
    title("📊 Data Analysis")
    c1, c2, c3 = st.columns(3)
    smoker = c1.multiselect("Smoker", df["smoker"].unique(), default=list(df["smoker"].unique()))
    sex = c2.multiselect("Gender", df["sex"].unique(), default=list(df["sex"].unique()))
    region = c3.multiselect("Region", df["region"].unique(), default=list(df["region"].unique()))
    data = df[df["smoker"].isin(smoker) & df["sex"].isin(sex) & df["region"].isin(region)]

    if data.empty:
        st.warning("No data for these filters. Please select at least one option in each filter.")
    else:
        st.subheader("Dataset preview")
        st.dataframe(data.head(10))
        a, b = st.columns(2)
        a.subheader("Data types")
        a.dataframe(data.dtypes.astype(str).rename("type"))
        b.subheader("Missing values")
        b.dataframe(data.isnull().sum().rename("missing"))
        st.subheader("Descriptive statistics")
        st.dataframe(data.describe())

        st.subheader("Charts")
        a, b = st.columns(2)
        a.plotly_chart(px.histogram(data, x="charges", nbins=40, title="Charges distribution"))
        b.plotly_chart(px.box(data, x="smoker", y="charges", color="smoker", title="Charges by smoker"))
        a.plotly_chart(px.box(data, x="region", y="charges", color="region", title="Charges by region"))
        b.plotly_chart(px.scatter(data, x="age", y="charges", color="smoker", title="Age vs charges"))
        st.plotly_chart(px.imshow(data.select_dtypes("number").corr(), text_auto=".2f",
                                  title="Correlation heatmap"))
    footer()

# ============ PAGE 3: MODEL PERFORMANCE ============
elif page == "🤖 Model Performance":
    title("🤖 Model Performance")
    st.success(f'Best model: **{info["best_model"]}** (lowest cross-validation RMSE)')
    st.dataframe(compare)
    st.download_button("⬇️ Download model comparison (CSV)", compare.to_csv(index=False), "model_comparison.csv")
    st.plotly_chart(px.bar(compare, x="CV RMSE", y="Model", orientation="h", color="Model",
                           title="Cross-validation RMSE (smaller is better)"))

    preds["Residual"] = preds["Actual"] - preds["Predicted"]
    a, b = st.columns(2)
    a.plotly_chart(px.scatter(preds, x="Actual", y="Predicted", title="Actual vs Predicted (test data)"))
    b.plotly_chart(px.scatter(preds, x="Predicted", y="Residual", title="Residual plot"))
    st.plotly_chart(px.histogram(preds, x="Residual", nbins=30, title="Residual distribution"))
    footer()

# ============ PAGE 4: PREDICTION ============
elif page == "💰 Prediction":
    title("💰 Insurance Charges Prediction")
    c1, c2, c3 = st.columns(3)
    age = c1.number_input("Age", min_value=18, max_value=100, step=1, key="in_age")
    sex = c2.selectbox("Gender", ["male", "female"], key="in_sex")
    bmi = c3.number_input("BMI", min_value=10.0, max_value=60.0, step=0.1, key="in_bmi")
    c4, c5, c6 = st.columns(3)
    children = c4.number_input("Children", min_value=0, max_value=10, step=1, key="in_children")
    smoker = c5.selectbox("Smoker", ["no", "yes"], key="in_smoker")
    region = c6.selectbox("Region", ["southwest", "southeast", "northwest", "northeast"], key="in_region")
    rate = st.number_input("USD to INR rate (change to today's rate)", min_value=1.0, value=88.0, step=0.5)

    b1, b2, _ = st.columns([1.5, 1, 4])
    predict = b1.button("🔮 Predict Insurance Charges")
    b2.button("🔄 Reset", on_click=reset_form)

    if predict:
        if not (18 <= age <= 100 and 10 <= bmi <= 60 and 0 <= children <= 10):
            st.error("Please enter valid values: age 18-100, BMI 10-60, children 0-10.")
        else:
            row = pd.DataFrame([{"age": age, "sex": sex, "bmi": bmi, "children": children,
                                 "smoker": smoker, "region": region}])
            usd = float(model.predict(row)[0])
            st.session_state["result"] = (usd, usd * rate)
            st.session_state["history"].append({
                "Time": datetime.now().strftime("%H:%M:%S"), **row.iloc[0].to_dict(),
                "Model": info["best_model"], "Predicted USD": round(usd, 2), "Predicted INR": round(usd * rate, 2)})

    if st.session_state["result"]:
        usd, inr = st.session_state["result"]
        st.markdown(f'<div class="result">Estimated charges: ₹ {inr:,.0f}<br>'
                    f'<span style="font-size:1rem">≈ $ {usd:,.0f} | Model: {info["best_model"]}</span></div>',
                    unsafe_allow_html=True)

    st.warning("This is only an estimate based on a dataset. It is NOT an actual insurance quote.")

    if st.session_state["history"]:
        st.subheader("Prediction history (this session)")
        history = pd.DataFrame(st.session_state["history"])
        st.dataframe(history)
        st.download_button("⬇️ Download predictions (CSV)", history.to_csv(index=False), "predictions.csv")
    footer()

# ============ PAGE 5: ABOUT ============
else:
    title("ℹ️ About Project")
    st.markdown("""
**Objective:** estimate medical insurance charges from personal details.

**Dataset:** insurance data with 7 columns (age, sex, bmi, children, smoker, region, charges).

**Algorithms:** Linear Regression, Polynomial Regression (degree 2 and 3), Decision Tree, KNN, Random Forest.

**Technologies:** Python, Pandas, Matplotlib, Seaborn, Plotly, Scikit-learn, Streamlit.

**Evaluation:** data split 80:20. Models compared with 5-fold cross-validation RMSE on training data.
The test data was used only once, for the best model.

**Limitations:** small dataset, charges are in US dollars (converted to ₹ with the rate you enter),
and the result is an estimate, not a real quote.

**Future work:** more features, gradient boosting, real Indian insurance data.
""")
    footer()

**Cell 32:** Create `requirements.txt` and `README.md`.

In [ ]:
Path("requirements.txt").write_text(
    f"streamlit>=1.50.0\npandas\nnumpy\nscikit-learn=={sklearn.__version__}\njoblib\nplotly\n")

Path("README.md").write_text("""# Medical Insurance Charges Prediction

Machine learning regression project with a Streamlit app.
Predicts medical insurance charges from age, sex, BMI, children, smoker and region.

Models: Linear, Polynomial (2 and 3), Decision Tree, KNN, Random Forest.

Run: pip install -r requirements.txt, then: streamlit run app.py

The prediction is an estimate, not an actual insurance quote.
""")

**Cell 33:** Start the app, then open http://localhost:8501 in your browser.

In [ ]:
app = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.headless", "true"])

**Cell 34:** Stop the app when you are finished.

In [ ]:
app.terminate()

## 9. Run, upload to GitHub, deploy

**Run in terminal**
```
cd medical_insurance_prediction
pip install -r requirements.txt
streamlit run app.py
```

**Upload to GitHub** (first create an empty repository on github.com)
```
git init
git add .
git commit -m "Insurance prediction project"
git branch -M main
git remote add origin https://github.com/Anitha123-allam/medical-insurance-prediction.git
git push -u origin main
```

**Deploy on Streamlit Community Cloud**
1. Open share.streamlit.io and sign in with GitHub.
2. Click **Create app**, choose your repository, branch `main`, main file `app.py`.
3. Click **Deploy**.